# Phase 2 — Data preparation and problem definition

Reads `data/raw/` (real Overpass fetch, 2026-09-19 — see `data/README.md`),
cleans it, reprojects to the analysis CRS, and writes `data/processed/`.
This is that phase's deliverable notebook, not a scratch file — see
`STUDY_LOG.md` "Working style for this repo".

**⚠️ Not yet executed.** Written and reviewed without running it — the
session that wrote it has no local access to run Jupyter against real data
(no geopandas install reachable there; execution had to happen on this
machine instead, where the pinned package and its geospatial dependencies
are actually installed — see `.venv`). **Run every cell top to bottom** and
read the printed output of the "Sanity checks" section before trusting
`data/processed/`. If a check fails, stop and fix it here rather than
letting a later phase silently consume bad data — per `STUDY_LOG.md`'s rule
against quietly working around a problem.


In [1]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

SOURCE_CRS = "EPSG:4326"
ANALYSIS_CRS = "EPSG:32635"  # WGS 84 / UTM zone 35N — see data/README.md


## 1. Load

Both files come from `scripts/overpass_to_geojson.py` (already converted
from the raw Overpass JSON also sitting in `data/raw/`). Load as-is first,
before any cleaning, so the "before" counts are real and not already
filtered.


In [2]:
hospitals_raw = gpd.read_file(RAW / "hospitals.geojson")
mahalle_raw = gpd.read_file(RAW / "mahalle_boundaries.geojson")

print(f"hospitals_raw:  {len(hospitals_raw):>4} features, CRS={hospitals_raw.crs}")
print(f"mahalle_raw:    {len(mahalle_raw):>4} features, CRS={mahalle_raw.crs}")


hospitals_raw:  1020 features, CRS=EPSG:4326
mahalle_raw:     964 features, CRS=EPSG:4326


## 2. Sanity checks against `data/README.md`

Every number here is an *expectation* recorded in `data/README.md`, not a
fact — confirm it against the real fetch, don't assume it. This block
turns each of the README's caveats into a printed, eyeballable check.


In [3]:
# --- caveat: source CRS must actually be EPSG:4326 ---
assert hospitals_raw.crs is not None and hospitals_raw.crs.to_epsg() == 4326, hospitals_raw.crs
assert mahalle_raw.crs is not None and mahalle_raw.crs.to_epsg() == 4326, mahalle_raw.crs
print("OK: both layers are EPSG:4326 as expected.")


OK: both layers are EPSG:4326 as expected.


In [4]:
# --- caveat 4 (data/README.md): amenity values should be exactly {hospital, clinic} ---
amenity_counts = hospitals_raw["amenity"].value_counts(dropna=False)
print("amenity value counts:")
print(amenity_counts)
unexpected = set(amenity_counts.index) - {"hospital", "clinic"}
if unexpected:
    print(f"\n⚠️  Unexpected amenity values found: {unexpected} — the regex\n"
          "    amenity~'hospital|clinic' is unanchored (data/README.md caveat 4).\n"
          "    Decide whether to keep or drop these before proceeding.")
else:
    print("\nOK: only hospital/clinic present, as expected.")


amenity value counts:
amenity
clinic      599
hospital    421
Name: count, dtype: int64

OK: only hospital/clinic present, as expected.


In [5]:
# --- caveat 6 (data/README.md): mahalle count and geometry-type mix ---
geom_types = mahalle_raw.geometry.geom_type.value_counts()
print(f"mahalle features: {len(mahalle_raw)} (expected ~964, confirmed 2026-09-19 against the live DB)")
print(f"geometry types:\n{geom_types}")

n = len(mahalle_raw)
if not (900 <= n <= 1050):
    print(f"\n⚠️  Count ({n}) is far from the ~964 expected in data/README.md.\n"
          "    Don't proceed silently — this could mean admin_level drifted back\n"
          "    to the wrong value, or a re-fetch pulled a stale/partial response.\n"
          "    See data/README.md caveat 6 and 'Fetched' section.")
else:
    print("\nOK: count is in the expected range.")


mahalle features: 964 (expected ~964, confirmed 2026-09-19 against the live DB)
geometry types:
Polygon         955
MultiPolygon      9
Name: count, dtype: int64

OK: count is in the expected range.


In [6]:
# --- null / missing geometry ---
print("null geometry — hospitals:", hospitals_raw.geometry.isna().sum())
print("null geometry — mahalle:  ", mahalle_raw.geometry.isna().sum())

# --- geometry validity (shapely) ---
invalid_mahalle = (~mahalle_raw.geometry.is_valid).sum()
print(f"invalid mahalle geometries (pre-fix): {invalid_mahalle}")
if invalid_mahalle:
    print("  -> will be repaired with .buffer(0) in the cleaning step below;\n"
          "     re-run this cell after that step to confirm 0 remain.")


null geometry — hospitals: 0
null geometry — mahalle:   0
invalid mahalle geometries (pre-fix): 0


In [7]:
# --- every mahalle should carry a real name (osm2geojson converter skips
#     nothing silently — see scripts/overpass_to_geojson.py's skip-reporting
#     — but confirm here rather than trusting that alone) ---
unnamed = mahalle_raw[mahalle_raw["name"].isna() | (mahalle_raw["name"].str.strip() == "")]
print(f"mahalle with no name: {len(unnamed)}")
if len(unnamed):
    display(unnamed[["osm_id"]])


mahalle with no name: 0


## 3. Clean

Minimal, deliberate cleaning only — per `data/README.md` "Regenerating":
drop null geometry, repair invalid rings, keep only the fields the analysis
needs (but keep `osm_id` on everything — it's the ODbL attribution/
traceability trail back to the source feature, required by `data/README.md`
"License").


In [8]:
hospitals = hospitals_raw[hospitals_raw.geometry.notna()].copy()
mahalle = mahalle_raw[mahalle_raw.geometry.notna()].copy()

# Repair invalid mahalle rings (self-intersections from OSM ring assembly)
# rather than silently keeping or silently dropping them.
still_invalid_before = (~mahalle.geometry.is_valid).sum()
if still_invalid_before:
    mahalle["geometry"] = mahalle.geometry.buffer(0)
still_invalid_after = (~mahalle.geometry.is_valid).sum()
print(f"invalid mahalle geometries: {still_invalid_before} -> {still_invalid_after}")
assert still_invalid_after == 0, "buffer(0) did not repair every geometry — investigate before continuing"

# Keep the fields the analysis actually uses, plus osm_id for attribution.
hospitals = hospitals[["osm_id", "osm_type", "amenity", "name", "geometry"]]
mahalle = mahalle[["osm_id", "osm_type", "admin_level", "name", "geometry"]]

print(f"hospitals: {len(hospitals_raw)} -> {len(hospitals)} after cleaning")
print(f"mahalle:   {len(mahalle_raw)} -> {len(mahalle)} after cleaning")


invalid mahalle geometries: 0 -> 0
hospitals: 1020 -> 1020 after cleaning
mahalle:   964 -> 964 after cleaning


## 4. Reproject to the analysis CRS

**Every layer, individually, asserted immediately after.** This is the
single most expensive mistake the Vienna sibling study made (three lost
N=20 batches) — see `STUDY_LOG.md` "Analysis conventions" → CRS.


In [9]:
hospitals_m = hospitals.to_crs(ANALYSIS_CRS)
mahalle_m = mahalle.to_crs(ANALYSIS_CRS)

assert hospitals_m.crs.to_epsg() == 32635
assert mahalle_m.crs.to_epsg() == 32635
print(f"OK: both layers reprojected to {ANALYSIS_CRS} and asserted.")

# Sanity check the reprojection landed somewhere real (İstanbul in UTM 35N
# is roughly x: 240,000–530,000 m, y: 4,520,000–4,620,000 m).
b = mahalle_m.total_bounds
print(f"mahalle bounds in {ANALYSIS_CRS}: {b}")
assert 100_000 < b[0] < 700_000 and 4_400_000 < b[1] < 4_700_000, (
    "Reprojected bounds look wrong for Istanbul in UTM 35N — investigate "
    "before trusting any distance computed from this."
)


OK: both layers reprojected to EPSG:32635 and asserted.
mahalle bounds in EPSG:32635: [ 581527.03586988 4519307.13216367  748500.91644713 4604146.37037856]


## 5. Centroids (for the centroid-based distance variant)

Computed **after** reprojection, in the metric CRS — a centroid computed in
EPSG:4326 (degrees) is not the same point as one computed in EPSG:32635
(metres), and only the metric one is valid input to a distance calculation.
See `STUDY_LOG.md` "Distance semantics".


In [10]:
mahalle_m["centroid"] = mahalle_m.geometry.centroid
# Every centroid should fall inside its own polygon for a simple polygon;
# for a MultiPolygon it may not (e.g. a split mahalle) — flag rather than assume.
outside = ~mahalle_m.apply(lambda r: r.geometry.contains(r["centroid"]), axis=1)
print(f"mahalle where the centroid falls outside the polygon: {outside.sum()}")
if outside.sum():
    display(mahalle_m.loc[outside, ["osm_id", "name", "osm_type"]])
    print("Expected for MultiPolygon/oddly-shaped mahalle — not a bug — but\n"
          "name them here so the paper's methodology section can too.")


mahalle where the centroid falls outside the polygon: 9


,osm_id,name,osm_type
60,7648841,Mimar Kemalettin Mahallesi,relation
105,7786497,Fatih Mahallesi,relation
171,8766277,Orhanlı Mahallesi,relation
436,9390850,Malkoçoğlu Mahallesi,relation
573,9446484,Şamlar Mahallesi,relation
598,9460393,Kınalıada Mahallesi,relation
602,9460397,Maden Mahallesi,relation
610,9462498,Esenkent Mahallesi,relation
919,9680170,Karaburun Mahallesi,relation


Expected for MultiPolygon/oddly-shaped mahalle — not a bug — but
name them here so the paper's methodology section can too.


## 6. Write `data/processed/`

Kept in the analysis CRS (EPSG:32635) since that's what every downstream
notebook will consume — re-derive EPSG:4326 for mapping/display only where
needed, don't round-trip through it as the stored format.


In [11]:
hospitals_out = hospitals_m.drop(columns=[c for c in ["centroid"] if c in hospitals_m.columns])
mahalle_out = mahalle_m.drop(columns=["centroid"])  # keep centroid computation in-notebook for now,
                                                      # not duplicated as a stored column

hospitals_out.to_file(PROCESSED / "hospitals.geojson", driver="GeoJSON")
mahalle_out.to_file(PROCESSED / "mahalle.geojson", driver="GeoJSON")

print(f"wrote {PROCESSED / 'hospitals.geojson'}: {len(hospitals_out)} features, CRS={hospitals_out.crs}")
print(f"wrote {PROCESSED / 'mahalle.geojson'}:   {len(mahalle_out)} features, CRS={mahalle_out.crs}")


wrote ../data/processed/hospitals.geojson: 1020 features, CRS=EPSG:32635
wrote ../data/processed/mahalle.geojson:   964 features, CRS=EPSG:32635


## 7. Summary — copy this cell's real output into `paper/PLAN.md`

`paper/PLAN.md`'s "Findings to fold into the paper's discussion/limitations"
section is empty — this is the first real finding to add there, using the
actual numbers this run printed, not the numbers in this markdown cell
(written before execution).


In [12]:
summary = {
    "fetched": "2026-09-19 (see data/README.md 'Fetched')",
    "hospitals_raw": len(hospitals_raw),
    "hospitals_by_amenity": hospitals_raw["amenity"].value_counts().to_dict(),
    "hospitals_after_cleaning": len(hospitals_out),
    "mahalle_raw": len(mahalle_raw),
    "mahalle_geometry_types": mahalle_raw.geometry.geom_type.value_counts().to_dict(),
    "mahalle_after_cleaning": len(mahalle_out),
    "mahalle_invalid_geometries_repaired": int(still_invalid_before),
    "mahalle_centroid_outside_polygon": int(outside.sum()),
    "analysis_crs": ANALYSIS_CRS,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "fetched": "2026-09-19 (see data/README.md 'Fetched')",
  "hospitals_raw": 1020,
  "hospitals_by_amenity": {
    "clinic": 599,
    "hospital": 421
  },
  "hospitals_after_cleaning": 1020,
  "mahalle_raw": 964,
  "mahalle_geometry_types": {
    "Polygon": 955,
    "MultiPolygon": 9
  },
  "mahalle_after_cleaning": 964,
  "mahalle_invalid_geometries_repaired": 0,
  "mahalle_centroid_outside_polygon": 9,
  "analysis_crs": "EPSG:32635"
}
